# Currency Detector — YOLOv8 Classification Training
Trains a YOLOv8n-cls model to classify ₹200 and ₹500 Indian currency notes.

**Runtime:** GPU (T4 or better) — Runtime → Change runtime type → T4 GPU

In [ ]:
# Cell 1: Install dependencies
!pip install ultralytics roboflow --quiet

In [ ]:
# Cell 2: Verify GPU is available
import torch
print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Cell 3: Download Indian currency dataset from Roboflow
# Go to https://universe.roboflow.com and search 'Indian currency'
# Find a dataset with 200 and 500 classes, click Download → YOLOv8 Classification
# Copy your API key and dataset code below

from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")  # <- paste your key here
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")  # <- paste project details
version = project.version(1)
dataset = version.download("folder")  # classification format

print('Dataset path:', dataset.location)

In [ ]:
# Cell 3b (ALTERNATIVE): Manual dataset — skip if using Roboflow above
# Upload a zip of images organized as:
#   dataset/train/200/*.jpg
#   dataset/train/500/*.jpg
#   dataset/val/200/*.jpg
#   dataset/val/500/*.jpg

import os
DATASET_PATH = "dataset"  # change if your zip extracted to a different folder

# Check structure
for split in ['train', 'val']:
    for cls in ['200', '500']:
        path = f"{DATASET_PATH}/{split}/{cls}"
        count = len(os.listdir(path)) if os.path.exists(path) else 0
        print(f"{path}: {count} images")

In [ ]:
# Cell 4: Train YOLOv8n-cls
from ultralytics import YOLO

DATASET_PATH = "dataset"  # update if using Roboflow: dataset.location

model = YOLO("yolov8n-cls.pt")  # pretrained backbone, fine-tune on currency

results = model.train(
    data=DATASET_PATH,
    epochs=30,
    imgsz=224,
    batch=32,
    name="currency_cls",
    patience=10,      # early stop if no improvement for 10 epochs
    augment=True,     # rotation, flip, brightness — helps with notes at angles
)

print("Training complete.")

In [ ]:
# Cell 5: Evaluate on validation set
metrics = model.val()
print(f"Top-1 accuracy: {metrics.top1:.2%}")
print(f"Top-5 accuracy: {metrics.top5:.2%}")

In [ ]:
# Cell 6: Quick test on a sample image
import glob
from pathlib import Path

DATASET_PATH = "dataset"
sample = glob.glob(f"{DATASET_PATH}/val/**/*.jpg", recursive=True)[0]
true_class = Path(sample).parent.name

pred = model.predict(sample, verbose=False)[0]
top1_idx = pred.probs.top1
top1_conf = float(pred.probs.top1conf)
pred_class = pred.names[top1_idx]

print(f"Sample: {sample}")
print(f"True class : ₹{true_class}")
print(f"Predicted  : ₹{pred_class} ({top1_conf:.0%} confidence)")

In [ ]:
# Cell 7: Download best.pt
import shutil
from google.colab import files

best_pt = "runs/classify/currency_cls/weights/best.pt"
shutil.copy(best_pt, "best.pt")
files.download("best.pt")
print("Downloaded best.pt — place it in model/ in your currency-detector project.")